# fbm3d Example 1: Basic ISM Density Field Generation

This notebook demonstrates how to:
1. Generate a 3D ISM density field using the fbm3d_ISM class
2. Compare two different algorithms (method=1 and method=2)
3. Visualize the density distribution (3D slices and histogram)
4. Analyze the power spectrum

The ISM (Interstellar Medium) density field is generated with a lognormal distribution characterized by Mach number and magnetic field parameter.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# IMPORTANT: Edit the following PATH variable according to your installation
PATH='/Users/kiseon/MoCafe/fbm3d'

import sys
sys.path.append(PATH)
from fbm_lib import *

## fbm3d_ISM: ISM Density Field Generation

**fbm3d_ISM** creates a 3D density field with:
- **Distribution**: Log-normal distribution
- **Spectrum**: Power-law-type power spectral density that mimics the interstellar medium

### Available Algorithms
- **method = 1**: Algorithm described in [Seon (2012)](https://ui.adsabs.harvard.edu/abs/2012ApJ...761L..17S/abstract)
  - Slower execution but reliable results
  
- **method = 2** (default): Implementation based on [Lewis & Austin (2002)](https://ams.confex.com/ams/11AR11CP/webprogram/Paper42772.html)
  - Faster computation with good quality results
  - Retains core ideas but differs in implementation details

### ISM Parameters
- `mach`: Mach number (typical value: 6.0) - characterizes turbulence strength
- `bvalue`: Magnetic field parameter (typical value: 0.4)
- The log-density standard deviation: `sigma_g = sqrt(ln(1 + (bvalue*mach)^2))`

In [2]:
# Grid resolution: 256^3
nx = 256
ny, nz = nx, nx

# Random seed control
seed = None                  # None: auto-generate; specify integer for reproducibility
#seed = 4070726807          # Example: uncomment to use specific seed
#seed = 2436634031

# ISM parameters
mach    = 6.0              # Mach number (turbulence strength)
bvalue  = 0.4              # Magnetic field parameter

# Calculate log-density standard deviation from ISM parameters
sigma_g = np.sqrt(np.log(1.0+(bvalue*mach)**2))
mean_g  = 0.0              # Mean of log(density) is typically 0

# Generate using method = 2 (Lewis & Austin 2002, default)
# If seed is None: random seed generated and saved as a.seed
# Log-density has mean=mean_g and standard deviation=sigma_g
a = fbm3d_ISM(nx=nx,ny=ny,nz=nz,mach=mach, seed=seed, verbose=True)

# Generate using method = 1 (Seon 2012) with SAME random seed for comparison
seed = a.seed              # Reuse seed from method=2 for fair comparison
p = fbm3d_ISM(nx=nx,ny=ny,nz=nz,mach=mach, method=1, seed=seed)

# Print results
print('<<== seed ==>> ', seed)
print(a.slope_ln, p.slope_ln)  # Compare power spectrum slopes

iteration =   1 / convergence = 43.32 %
iteration =   2 / convergence = 75.72 %
iteration =   3 / convergence = 90.56 %
iteration =   4 / convergence = 96.82 %
iteration =   5 / convergence = 99.34 %
iteration =   6 / convergence = 99.72 %
<<== seed ==>>  345684308
2.860360523520207 2.860360523520207


## Visualization: 3D Density Slices and Histogram

The figure shows:
- **Left 3 panels**: Three orthogonal slices (xy, xz, yz) of the 3D log-density field
  - Each slice shows spatial structure of density variations
  - Color scale represents log(rho)
- **Right panel**: Histogram of log-density values
  - Blue: actual distribution from generated field
  - Red: theoretical Gaussian (expected distribution)
  - Should align if algorithm working correctly

In [3]:
# Plot Density Field from method = 2 (default method, Lewis & Austin 2002)

fig, ax = plt.subplots(1,4, figsize=(24,4))
ln_data = np.log(a.data)  # Convert to log scale for better visualization

# Plot three orthogonal slices through center of 3D volume
im0 = ax[0].imshow(ln_data[:,:,nz//2])      # xy-plane slice at z=nz/2
im1 = ax[1].imshow(ln_data[:,ny//2,:])      # xz-plane slice at y=ny/2
im2 = ax[2].imshow(ln_data[nx//2,:,:])      # yz-plane slice at x=nx/2

# Add colorbars to each slice
add_colorbar(im0)
add_colorbar(im1)
add_colorbar(im2)

# Plot histogram of log(density) values
bins = np.linspace(-5.0*sigma_g, 5.0*sigma_g, 100) + mean_g  # Histogram bins
x    = (bins[:-1]+bins[1:])/2.0                               # Bin centers
h,_  = np.histogram(ln_data, bins, density=True)              # Normalized histogram

# Theoretical Gaussian distribution for comparison
y    = 1./np.sqrt(2.*np.pi*sigma_g**2) * np.exp(-(x-mean_g)**2/(2*sigma_g**2))

ax[3].plot(x,h)                              # Histogram from data (blue)
ax[3].plot(x,y,color='red')                  # Theoretical curve (red)
ax[3].set_xlabel(r'$\ln\rho_{\rm lewis}$')
ax[3].set_ylabel(r'$P(\ln\rho)$')

Text(0, 0.5, '$P(\\ln\\rho)$')

## Comparison: Density Field from method=1 (Seon 2012)

Compare density field generated using method=1 (Seon 2012) with method=2.
Both use same random seed for direct comparison.

In [4]:
# Plot Density Field from method = 1 (Seon 2012)

fig, ax = plt.subplots(1,4, figsize=(24,4))
ln_data = np.log(p.data)  # Convert to log scale
im0 = ax[0].imshow(ln_data[:,:,nz//2])
im1 = ax[1].imshow(ln_data[:,ny//2,:])
im2 = ax[2].imshow(ln_data[nx//2,:,:])

add_colorbar(im0)
add_colorbar(im1)
add_colorbar(im2)

bins = np.linspace(-5.0*sigma_g, 5.0*sigma_g, 100) + mean_g
x    = (bins[:-1]+bins[1:])/2.0
h,_  = np.histogram(ln_data, bins, density=True)
y    = 1./np.sqrt(2.*np.pi*sigma_g**2) * np.exp(-(x-mean_g)**2/(2*sigma_g**2))
ax[3].plot(x,h)
ax[3].plot(x,y,color='red')
ax[3].set_xlabel(r'$\ln\rho_{\rm seon}$')
ax[3].set_ylabel(r'$P(\ln\rho)$')

Text(0, 0.5, '$P(\\ln\\rho)$')